# Replay protection experiment analysis

This notebook delegates all validation and metric recomputation to `analyze_replay_protection.py`. It refuses to present a comparison when the 12-configuration matrix, request observations, protected-operation evidence, or recorded summary is incomplete or inconsistent.

The experiment directory below is explicit. Replace it with the path printed by the latest full run when analyzing another dataset.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

project_root = Path.cwd()
if not (project_root / "run-local-demo-2.js").exists():
    project_root = project_root.parent
experiment_dir = project_root / "results/replay-2026-11-12T20-09-58-896Z-c0128909"
analysis_script = project_root / "analysis" / "analyze_replay_protection.py"

result = subprocess.run(
    [sys.executable, str(analysis_script), str(experiment_dir)],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode:
    raise RuntimeError("Experiment validation failed; do not compare this dataset.")

analysis_dir = experiment_dir / "analysis"
validation = json.loads((analysis_dir / "validation_report.json").read_text())
assert validation["valid"] is True


In [ ]:
import pandas as pd
from IPython.display import display, Image

validation_rows = pd.read_csv(analysis_dir / "scenario_validation.csv")
derived_metrics = pd.read_csv(analysis_dir / "derived_metrics.csv")
method_comparison = pd.read_csv(analysis_dir / "method_comparison.csv")
requests = pd.read_csv(experiment_dir / "requests.csv")
summary = pd.read_csv(experiment_dir / "summary.csv")

print(f"Validated {validation['valid_scenarios']} of {validation['scenario_rows']} scenario/repetition rows")
display(validation_rows)
display(method_comparison)


In [ ]:
display(Image(filename=str(analysis_dir / "security_rates.png")))
display(Image(filename=str(analysis_dir / "latency_by_method.png")))


## Metric definitions

- Replay detection rate: detected serial replays plus detected concurrent duplicate attempts, divided by serial replay attempts plus (concurrent batch size minus one) per batch.
- False acceptance rate: accepted serial replay attempts plus concurrent acceptances beyond the first per `jti`, divided by that same replay-attempt denominator.
- Legitimate acceptance and false rejection: accepted or non-accepted observed legitimate first-use/performance requests divided by the number of those observed requests.
- Cross-replica rate: detected cross-replica replay observations divided by observed cross-replica replay requests; one-replica cases are not applicable.
- Latency percentiles and replay-store latency are recomputed from measured request rows. Throughput is completed performance requests divided by their observed wall-clock span.
- Protected-operation count includes only rows whose explicit Order Service response header confirms the protected handler executed. Missing evidence remains missing and invalidates the run.
